## 6.1 Univariable Plots
### 6.1.1 Histograms

In [ ]:
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import metpy.calc as mpcalc
from metpy.plots import SkewT
from metpy.units import units

In [ ]:
plt.rcParams.update({"font.size": 16, "figure.figsize": (6, 6), "font.sans-serif": ["Segoe UI", "DejaVu Sans", "sans-serif"]})

In [ ]:
filename = "hms_fire20250109.txt"
fires_path = Path("./data/txt") / filename
fires = pd.read_csv(fires_path, skipinitialspace=True)
fires.head()

In [ ]:
bins_10 = np.arange(0, 1000, 10)

In [ ]:
plt.figure()
plt.hist(fires["FRP"], bins=bins_10)
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)
ax.hist(fires["FRP"], bins=bins_10)
ax.set_yscale("log")
ax.set_xlabel("Fire Radiative Power (MW)")
ax.set_ylabel("Counts")
plt.show()

### 6.1.2 Barplots

In [ ]:
ranges = [0.0, 100.0, 200.0, 1000.0]
categories = ["Low", "Medium", "High"]

intensity = pd.cut(fires["FRP"], bins=ranges, include_lowest=True, \
    labels=categories)
intensity.head()

In [ ]:
intensity_counts = intensity.value_counts()
intensity_counts

In [ ]:
plt.figure()
ax = plt.subplot(111)
ax.bar(x=intensity_counts.index, height=intensity_counts)
ax.set_yscale("log")
ax.set_xlabel("Fire Intensity")
ax.set_ylabel("Counts")
plt.show()

## 6.2 Two Variable Plots

In [ ]:
filename = "20250101_20250131_CalTech.lev20"
aeronet_file = Path("./data/txt") / filename
aeronet = pd.read_csv(aeronet_file, skiprows=6)

In [ ]:
aeronet.head()

### 6.2.1 Converting Data to a Time Series

In [ ]:
aeronet['Date(dd:mm:yyyy)'].head()

In [ ]:
aeronet["datetime"] = aeronet["Date(dd:mm:yyyy)"] + " " + aeronet["Time(hh:mm:ss)"]
aeronet["datetime"].head()

In [ ]:
# aeronet.sort_values(by="Date(dd:mm:yyyy)", inplace=True)
# aeronet["datetime"].values[0], aeronet["datetime"].values[-1]

In [ ]:
aeronet["datetime"].head()

In [ ]:
fmt = "%d:%m:%Y %H:%M:%S"
aeronet["datetime"] = pd.to_datetime(aeronet["datetime"], format=fmt)
aeronet["datetime"].head()

### 6.2.2 Useful Plot Customizations

In [ ]:
xlims = [datetime.date(2025, 1, 1), datetime.date(2025, 1, 31)]
ylims = [0, 3]
ylabel = "Aerosol Optical Depth (AOD) at 500 nm"

In [ ]:
# Options to print figures into notebook/increase size
plt.rcParams["figure.figsize"] = [12, 6]
plt.rcParams.update({"font.size": 16})

In [ ]:
plt.rcParams.update({"font.size": 16, 
    "figure.figsize": [12, 6],
    "font.sans-serif": ["Segoe UI", "DejaVu Sans", "sans-serif"]
    })

### 6.2.3 Scatter Plots

In [ ]:
fig = plt.figure()
ax = plt.subplot(111)

ax.scatter(aeronet["datetime"], aeronet["AOD_500nm"], c="lightgray")
plt.xticks(rotation=35)
ax.set_xlim(xlims[0], xlims[1])
ax.set_ylim(ylims[0], ylims[1])

ax.set_ylabel(ylabel)
plt.show()

### 6.2.4 Line Plots

In [ ]:
aeronet_filtered = aeronet[aeronet["AOD_500nm"] > -999.0]
aeronet_daily_average = aeronet_filtered \
    .resample("D", on="datetime") \
    .mean(numeric_only=True)
aeronet_daily_average["AOD_500nm"]

In [ ]:
aeronet_daily_average = aeronet_daily_average.reset_index()

In [ ]:
fig = plt.figure()
ax = plt.subplot(111)

ax.scatter(aeronet["datetime"], aeronet["AOD_500nm"], c="lightgray")
ax.plot(aeronet_daily_average["datetime"], \
    aeronet_daily_average["AOD_500nm"], linewidth=3)

ax.set_xlim(xlims[0], xlims[1])
ax.set_ylim(ylims[0], ylims[1])

ax.set_ylabel(ylabel)

fig.autofmt_xdate()

plt.show()

### 6.3.5 Adding data to an existing plot

In [ ]:
fig = plt.figure()
ax = plt.subplot(111)

ax.plot(aeronet_daily_average["datetime"], aeronet_daily_average["AOD_1640nm"], label="1640 nm")
ax.plot(aeronet_daily_average["datetime"], aeronet_daily_average["AOD_1020nm"], label="1020 nm")
ax.plot(aeronet_daily_average["datetime"], aeronet_daily_average["AOD_500nm"], label="500 nm")

ax.legend(loc="upper right")

ax.set_xlim(xlims[0], xlims[1])
ax.set_ylim(0, 1)
ax.set_ylabel(ylabel)

fig.autofmt_xdate()

plt.show()

### 6.3.6 Plotting two side-by-side plots

In [ ]:
fig, ax = plt.subplots(ncols=2)

ax[0].set_title("500 nm")
ax[0].plot(aeronet_daily_average["datetime"], aeronet_daily_average["AOD_500nm"], c="green")
ax[0].set_ylabel("AOD")

ax[1].set_title("1640 nm")
ax[1].plot(aeronet_daily_average["datetime"], aeronet_daily_average["AOD_1640nm"], c="blue")

for ax0 in ax:
    ax0.set_xlim(xlims[0], xlims[1])
    ax0.set_ylim(0, 1)

fig.autofmt_xdate()
plt.show()

### 6.3.7 Skew-T Log-P

In [ ]:
filename = "NUCAPS-EDR_v3r2_j01_s202501092048589_e202501092049287_c202501092139200.nc"
nucaps_file = Path("data/nucaps") / filename
nucaps = xr.open_dataset(nucaps_file, engine="h5netcdf")

list(nucaps.H2O_MR.attrs)

In [ ]:
nucaps["H2O_MR"].units, nucaps["Temperature"].units, nucaps["Pressure"].units

In [ ]:
nucaps["H2O_MR"].dims

In [ ]:
profile = nucaps.sel(Number_of_CrIS_FORs=0)

In [ ]:
profile["H2O_MR"].dims

In [ ]:
T = profile.Temperature.values.flatten()*units("K")
MR = profile.H2O_MR.values.flatten()*units("kg/kg")
P = profile.Effective_Pressure.values.flatten()*units("millibar")

In [ ]:
RH = mpcalc.relative_humidity_from_mixing_ratio(P, T, MR)
Td = mpcalc.dewpoint_from_relative_humidity(T, RH)

In [ ]:
RH[RH < 0] = 0
Td = mpcalc.dewpoint_from_relative_humidity(T, RH)

In [ ]:
fig = plt.figure(figsize=[8,8])
skew = SkewT(fig, subplot=111)

skew.plot(P, T, label="Temperature")
skew.plot(P, Td, linestyle="--", label="Dewpoint")
skew.ax.legend(loc="upper right")
plt.show()

## 6.3 Three Variable Plots
### 6.3.1 Filled contour

In [ ]:
filename = "HAQ_TROPOMI_NO2_GLOBAL_QA75_L3_Monthly_082025_V2.4_20250912.nc4"
tropomi_file = Path("data/tropomi") / filename
tropomi = xr.open_dataset(tropomi_file, engine="h5netcdf")

In [ ]:
tropomi

In [ ]:
tropomi_no2 = tropomi["Tropospheric_NO2"]
tropomi_lat = tropomi["Latitude"]
tropomi_lon = tropomi["Longitude"]

In [ ]:
X_no2, Y_no2 = np.meshgrid(tropomi_lon, tropomi_lat)

In [ ]:
tropomi_no2.shape, X_no2.shape

In [ ]:
levels = [0, 8e14, 9e14, 1e15, 2e15, 3e15, 4e15, 5e15, 6e15]

fig, ax = plt.subplots(figsize=[12, 8])
no2_plot = ax.contourf(X_no2, Y_no2, tropomi_no2, levels=levels)
fig.colorbar(no2_plot, orientation="horizontal", ax=ax, shrink=0.8)

ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(True)

plt.show()

In [ ]:
fires

In [ ]:
fires["lon_bin"] = pd.cut(fires["Lon"], bins=20)
fires["lat_bin"] = pd.cut(fires["Lat"], bins=20)

fire_count = fires.groupby(["lon_bin", "lat_bin"], observed=False).size()

In [ ]:
fire_count = fire_count.reset_index(name="Count")
fire_count.head()

In [ ]:
fire_count["lon_mid"] = fire_count["lon_bin"].apply(lambda x: x.mid)
fire_count["lat_mid"] = fire_count["lat_bin"].apply(lambda x: x.mid)

In [ ]:
fire_count_3D = fire_count.pivot(index="lat_mid", columns="lon_mid", values="Count")
fire_count_3D

In [ ]:
X_fires, Y_fires = np.meshgrid(fire_count_3D.columns, \
    fire_count_3D.index)

In [ ]:
levels = np.arange(0, 2775, 200)

fig, ax = plt.subplots(figsize=[12, 8])
fire_plot = ax.contourf(X_fires, Y_fires, fire_count_3D, cmap="plasma", levels=levels)
fig.colorbar(fire_plot, orientation="horizontal", ax=ax, shrink=0.8)

ax.scatter(fires["Lon"], fires["Lat"], s=1, c="yellow", alpha=0.2)

ax.set_xlim(-160, -60)
ax.set_ylim(10, 60)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(True)

plt.show()

### 6.3.2 Mesh plots

In [ ]:
fig, ax = plt.subplots(figsize=[12, 8])

co_plot = ax.pcolormesh(X_no2, Y_no2, tropomi_no2, vmin=0, vmax=5e15)
fig.colorbar(co_plot, orientation="horizontal", ax=ax, shrink=0.8)

ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

ax.grid(True)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=[12,8])

fire_plot = ax.pcolormesh(X_fires, Y_fires, fire_count_3D, cmap="plasma", norm=colors.LogNorm(vmin=1, vmax=1000, clip=True))
fig.colorbar(fire_plot, orientation="horizontal", shrink=1)
ax.scatter(fires["Lon"], fires["Lat"], s=1, c="black", alpha=0.2)

ax.set_xlim(-160, -60)
ax.set_ylim(10, 60)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(True)
plt.show()

In [ ]:
# fig, ax = plt.subplots(figsize=[12,8])

# fire_plot = ax.pcolormesh(X_fires, Y_fires, fire_count_3D, cmap="plasma", norm=colors.LogNorm(vmin=1, vmax=1000, clip=False))
# fig.colorbar(fire_plot, orientation="horizontal", shrink=1)
# ax.scatter(fires["Lon"], fires["Lat"], s=1, c="black", alpha=0.2)

# ax.set_xlim(-160, -60)
# ax.set_ylim(10, 60)
# ax.set_xlabel("Longitude")
# ax.set_ylabel("Latitude")
# ax.grid(True)
# plt.show()